# Clean movie metadata

Turn the joined Netflix–TMDB table (`movies_metadata.csv`) into a raw movie table for later EDA. Keep every matched movie, drop unnecessary columns, remove movies with missing (zero) budget, revenue or runtime,  and parse list fields.
Save the result to `data/processed/movies_cleaned.csv`. Feature engineering happens in `05_movie_features.ipynb`.

## 1. Load metadata

Read the joined Netflix–TMDB movie table.

In [1]:
from pathlib import Path
import ast
import json
import pandas as pd

ROOT = Path("..")
movies = pd.read_csv(ROOT / "data/processed/movies_metadata.csv")
print(movies.shape)
movies.head(2)

(1806, 25)


,netflix_movie_id,tmdb_id,netflix_title,tmdb_title,year,budget,genres,homepage,keywords,original_language,...,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,cast,crew
0,3,17139,Character,Character,1997.0,4500000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10769, ""n...",NaN,"[{""id"": 3683, ""name"": ""law and ethics""}]",nl,...,1997-04-17,0,122.0,"[{""iso_639_1"": ""nl"", ""name"": ""Nederlands""}, {""...",Released,NaN,7.7,25,"[{""cast_id"": 4, ""character"": ""Dreverhaven"", ""c...","[{""credit_id"": ""52fe47099251416c7508bca3"", ""de..."
1,30,6964,Something's Gotta Give,Something's Gotta Give,2003.0,80000000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 35, ""name...",NaN,"[{""id"": 965, ""name"": ""age difference""}, {""id"":...",en,...,2003-12-12,266728738,128.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,Schmucks are people too.,6.3,410,"[{""cast_id"": 1, ""character"": ""Harry Sanborn"", ...","[{""credit_id"": ""548200699251416e7800666d"", ""de..."


## 2. Drop unused columns and unify title

Drop sparse or unused text fields. Keep Netflix id and a single title.

In [2]:
DROP_COLS = [
    "homepage",
    "tagline",
    "keywords",
    "status",
    "popularity",
    "vote_average",
    "vote_count",
    "tmdb_title",
    "tmdb_id",
    "original_title",
    "overview",
    "release_date"
]

df = movies.drop(columns=DROP_COLS).rename(columns={"netflix_title": "title"})
print(df.shape)
print(list(df.columns))

(1806, 13)
['netflix_movie_id', 'title', 'year', 'budget', 'genres', 'original_language', 'production_companies', 'production_countries', 'revenue', 'runtime', 'spoken_languages', 'cast', 'crew']


## 3. Remove movies with zero budget, revenue, or runtime

TMDB encodes unknown budget/revenue/runtime as 0. Drop any row where budget, revenue, or runtime is zero so later numeric features use only observed production values.

In [3]:
n_before = len(df)
zero_mask = (df["budget"] == 0) | (df["revenue"] == 0) | (df["runtime"] == 0)
df = df[~zero_mask].copy()
print(f"removed {n_before - len(df)} rows with budget/revenue/runtime == 0; remaining {len(df)}")
print(
    "zeros left - budget:",
    int((df["budget"] == 0).sum()),
    "revenue:",
    int((df["revenue"] == 0).sum()),
    "runtime:",
    int((df["runtime"] == 0).sum()),
)

removed 484 rows with budget/revenue/runtime == 0; remaining 1322
zeros left - budget: 0 revenue: 0 runtime: 0


## 4. Parse JSON fields

Parse list-of-dict string columns into name lists. Extract the director and cast, then drop the raw JSON columns.


In [4]:
def parse_names(val):
    if pd.isna(val) or val == "":
        return []
    try:
        obj = ast.literal_eval(val) if isinstance(val, str) else val
    except (ValueError, SyntaxError):
        return []
    if not isinstance(obj, list):
        return []
    return [x.get("name") for x in obj if isinstance(x, dict) and x.get("name")]


def extract_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str) if isinstance(crew_str, str) else crew_str
    except (ValueError, SyntaxError):
        return None
    if not isinstance(crew, list):
        return None
    for person in crew:
        if isinstance(person, dict) and person.get("job") == "Director":
            return person.get("name")
    return None


df["genre_names"] = df["genres"].map(parse_names)
df["company_names"] = df["production_companies"].map(parse_names)
df["country_names"] = df["production_countries"].map(parse_names)
df["language_names"] = df["spoken_languages"].map(parse_names)
df["cast_names"] = df["cast"].map(parse_names)  # full cast for film counts
df["director"] = df["crew"].map(extract_director)

print("directors missing:", int(df["director"].isna().sum()))
print("cast name list lengths (sample):", df["cast_names"].map(len).describe()[["mean", "50%", "max"]].to_dict())
df[["title", "cast_names"]].head(3)

directors missing: 0
cast name list lengths (sample): {'mean': 23.876701966717096, '50%': 18.0, 'max': 213.0}


,title,cast_names
1,Something's Gotta Give,"[Jack Nicholson, Diane Keaton, Keanu Reeves, F..."
2,Jade,"[David Caruso, Linda Fiorentino, Chazz Palmint..."
4,Dragonheart,"[Dennis Quaid, David Thewlis, Pete Postlethwai..."


In [5]:
df.columns

Index(['netflix_movie_id', 'title', 'year', 'budget', 'genres',
       'original_language', 'production_companies', 'production_countries',
       'revenue', 'runtime', 'spoken_languages', 'cast', 'crew', 'genre_names',
       'company_names', 'country_names', 'language_names', 'cast_names',
       'director'],
      dtype='object')

In [6]:
df = df.drop(columns = ['genres', 'production_companies', 'production_countries', 'crew',
                        'cast', 'spoken_languages'])

In [7]:
LIST_COLS = ["genre_names", "company_names", "country_names", "language_names", "cast_names"]
for col in LIST_COLS:
    df[col] = df[col].map(json.dumps)

out_path = ROOT / "data/processed/movies_cleaned.csv"
df.to_csv(out_path, index=False)
print(f"saved {out_path} ({df.shape[0]} rows x {df.shape[1]} cols)")

saved ..\data\processed\movies_cleaned.csv (1322 rows x 13 cols)
